In [ ]:
# =============================================================================
# CELL TO SAVE ALL MODELS FOR DEPLOYMENT
# Run this cell to save trained models with preprocessors to models/ folder
# =============================================================================

import joblib
import json


print("=" * 60)
print("SAVING MODELS FOR DEPLOYMENT")
print("=" * 60)

# Feature names after transformation
num_feature_names = ['Customer_Age', 'Months_on_book', 'Total_Relationship_Count',
'Months_Inactive_12_mon',
                    'Contacts_Count_12_mon', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1',
'Total_Trans_Amt',
                    'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio',
'engagement_score']

# Get categorical feature names from the fitted preprocessor
cat_encoder = preprocessor.named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(['Gender', 'Education_Level', 'Marital_Status',
'Income_Category', 'Card_Category']).tolist()

# All feature names in order
all_feature_names = num_feature_names + cat_feature_names

# Save feature names
with open('models/features_in_order.json', 'w') as f:
    json.dump(all_feature_names, f, indent=2)
print(f"\n✅ Saved features_in_order.json ({len(all_feature_names)} features)")

# Re-train all models on full training data and save as bundles
# (Models were already trained above, reusing them)

# ============ 1. STACKING MODEL (BEST - XGB Meta) ============
print("\n[1/6] Saving Stacking (XGB Meta) - BEST MODEL...")

# Transform training data
X_train_eng = engineer_features(X_train)
X_train_transformed = preprocessor.fit_transform(X_train_eng)

# Refit stacking model on transformed data
stack_est_final = [
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_split=5,
class_weight='balanced', random_state=42)),
    ('gb', HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3,
class_weight='balanced', random_state=42)),
    ('xgb', XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8,
colsample_bytree=0.8,
                        scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train),
eval_metric='logloss', random_state=42))
]

stacking_xgb_final = StackingClassifier(
    estimators=stack_est_final,
    final_estimator=XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                    scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train),
                                    eval_metric='logloss', random_state=42),
    cv=5, stack_method='predict_proba', passthrough=False, n_jobs=-1
)
stacking_xgb_final.fit(X_train_transformed, y_train)

# Evaluate on test set
X_test_eng = engineer_features(X_test)
X_test_transformed = preprocessor.transform(X_test_eng)
y_pred_stack = stacking_xgb_final.predict(X_test_transformed)
y_prob_stack = stacking_xgb_final.predict_proba(X_test_transformed)[:, 1]

from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score
stack_metrics = {
    'recall': recall_score(y_test, y_pred_stack),
    'precision': precision_score(y_test, y_pred_stack),
    'f1': f1_score(y_test, y_pred_stack),
    'roc_auc': roc_auc_score(y_test, y_prob_stack)
}

# Save as bundle (preprocessor + model + metadata)
stacking_bundle = {
    'model': stacking_xgb_final,
    'preprocessor': preprocessor,
    'feature_names': all_feature_names,
    'numeric_features': ['Customer_Age', 'Months_on_book', 'Total_Relationship_Count',
'Months_Inactive_12_mon',
                        'Contacts_Count_12_mon', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1',
'Total_Trans_Amt',
                        'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'],
    'categorical_features': ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category',
'Card_Category'],
    'threshold': 0.5,
    'metrics': stack_metrics
}
joblib.dump(stacking_bundle, 'models/stacking_model.pkl')
print(f"✅ Saved models/stacking_model.pkl (Recall: {stack_metrics['recall']:.4f}, AUC: {stack_metrics['roc_auc']:.4f})")

# ============ 2. RANDOM FOREST ============
print("\n[2/6] Saving Random Forest...")
rf_final = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_split=5,
class_weight='balanced', random_state=42)
rf_final.fit(X_train_transformed, y_train)

y_pred_rf = rf_final.predict(X_test_transformed)
y_prob_rf = rf_final.predict_proba(X_test_transformed)[:, 1]
rf_metrics = {
    'recall': recall_score(y_test, y_pred_rf),
    'precision': precision_score(y_test, y_pred_rf),
    'f1': f1_score(y_test, y_pred_rf),
    'roc_auc': roc_auc_score(y_test, y_prob_rf)
}

rf_bundle = {
    'model': rf_final,
    'preprocessor': preprocessor,
    'feature_names': all_feature_names,
    'numeric_features': ['Customer_Age', 'Months_on_book', 'Total_Relationship_Count',
'Months_Inactive_12_mon',
                        'Contacts_Count_12_mon', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1',
'Total_Trans_Amt',
                        'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'],
    'categorical_features': ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category',
'Card_Category'],
    'threshold': 0.5,
    'metrics': rf_metrics
}
joblib.dump(rf_bundle, 'models/rf_model.pkl')
print(f"    ✅ Saved models/rf_model.pkl (Recall: {rf_metrics['recall']:.4f}, AUC:{rf_metrics['roc_auc']:.4f})")


# ============ 3. XGBOOST ============
print("\n[3/6] Saving XGBoost...")
xgb_final = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8,
colsample_bytree=0.8,
                        scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train),
eval_metric='logloss', random_state=42)
xgb_final.fit(X_train_transformed, y_train)

y_pred_xgb = xgb_final.predict(X_test_transformed)
y_prob_xgb = xgb_final.predict_proba(X_test_transformed)[:, 1]
xgb_metrics = {
    'recall': recall_score(y_test, y_pred_xgb),
    'precision': precision_score(y_test, y_pred_xgb),
    'f1': f1_score(y_test, y_pred_xgb),
    'roc_auc': roc_auc_score(y_test, y_prob_xgb)
}

xgb_bundle = {
    'model': xgb_final,
    'preprocessor': preprocessor,
    'feature_names': all_feature_names,
    'numeric_features': ['Customer_Age', 'Months_on_book', 'Total_Relationship_Count',
'Months_Inactive_12_mon',
                        'Contacts_Count_12_mon', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1',
'Total_Trans_Amt',
                        'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'],
    'categorical_features': ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category',
'Card_Category'],
    'threshold': 0.5,
    'metrics': xgb_metrics
}
joblib.dump(xgb_bundle, 'models/xgb_model.pkl')
print(f" ✅ Saved models/xgb_model.pkl (Recall: {xgb_metrics['recall']:.4f}, AUC:{xgb_metrics['roc_auc']:.4f})")


# ============ 4. GRADIENT BOOSTING ============
print("\n[4/6] Saving Gradient Boosting...")
gb_final = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3,
class_weight='balanced', random_state=42)
gb_final.fit(X_train_transformed, y_train)

y_pred_gb = gb_final.predict(X_test_transformed)
y_prob_gb = gb_final.predict_proba(X_test_transformed)[:, 1]
gb_metrics = {
    'recall': recall_score(y_test, y_pred_gb),
    'precision': precision_score(y_test, y_pred_gb),
    'f1': f1_score(y_test, y_pred_gb),
    'roc_auc': roc_auc_score(y_test, y_prob_gb)
}

gb_bundle = {
    'model': gb_final,
    'preprocessor': preprocessor,
    'feature_names': all_feature_names,
    'numeric_features': ['Customer_Age', 'Months_on_book', 'Total_Relationship_Count',
'Months_Inactive_12_mon',
                        'Contacts_Count_12_mon', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1',
'Total_Trans_Amt',
                        'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'],
    'categorical_features': ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category',
'Card_Category'],
    'threshold': 0.5,
    'metrics': gb_metrics
}
joblib.dump(gb_bundle, 'models/gb_model.pkl')
print(f"    ✅ Saved models/gb_model.pkl (Recall: {gb_metrics['recall']:.4f}, AUC:{gb_metrics['roc_auc']:.4f})")


# ============ 5. CATBOOST ============
print("\n[5/6] Saving CatBoost...")
cat_final = CatBoostClassifier(iterations=200, learning_rate=0.05, depth=4, random_state=42, verbose=0)
cat_final.fit(X_train_transformed, y_train)

y_pred_cat = cat_final.predict(X_test_transformed)
y_prob_cat = cat_final.predict_proba(X_test_transformed)[:, 1]
cat_metrics = {
    'recall': recall_score(y_test, y_pred_cat),
    'precision': precision_score(y_test, y_pred_cat),
    'f1': f1_score(y_test, y_pred_cat),
    'roc_auc': roc_auc_score(y_test, y_prob_cat)
}

cat_bundle = {
    'model': cat_final,
    'preprocessor': preprocessor,
    'feature_names': all_feature_names,
    'numeric_features': ['Customer_Age', 'Months_on_book', 'Total_Relationship_Count',
'Months_Inactive_12_mon',
                        'Contacts_Count_12_mon', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1',
'Total_Trans_Amt',
                        'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'],
    'categorical_features': ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category',
'Card_Category'],
    'threshold': 0.5,
    'metrics': cat_metrics
}
joblib.dump(cat_bundle, 'models/catboost_model.pkl')
print(f"    ✅ Saved models/catboost_model.pkl (Recall: {cat_metrics['recall']:.4f}, AUC:{cat_metrics['roc_auc']:.4f})")


# ============ 6. SAVE FEATURE ENGINEERING FUNCTION ============
print("\n[6/6] Saving feature engineering function...")

# Save as pickle so it can be loaded directly
joblib.dump(engineer_features, 'models/feature_engineering_func.pkl')

# Also save as Python module
feature_engine_code = '''"""
Feature engineering module for ChurnShield model.
Import this module before loading the saved models.
"""

def engineer_features(X_df):
    """
    Add derived features to the input dataframe.

    Parameters
    ----------
    X_df : pd.DataFrame
        Input dataframe with raw features

    Returns
    -------
    pd.DataFrame
        DataFrame with additional engineered features
    """
    df_eng = X_df.copy()
    df_eng['avg_amt_per_txn'] = df_eng['Total_Trans_Amt'] / (df_eng['Total_Trans_Ct'] + 1e-9)
    df_eng['engagement_score'] = df_eng['Total_Trans_Ct'] * df_eng['Avg_Utilization_Ratio']
    return df_eng
'''

with open('models/feature_engineering.py', 'w') as f:
    f.write(feature_engine_code)

print("    ✅ Saved models/feature_engineering_func.pkl")
print("    ✅ Saved models/feature_engineering.py")

# ============ SUMMARY ============
print("\n" + "=" * 60)
print("ALL MODELS SAVED SUCCESSFULLY")
print("=" * 60)
print("""
Files created in models/:
1. stacking_model.pkl - Best model (Stacking with XGB meta-learner)
2. rf_model.pkl - Random Forest model
3. xgb_model.pkl - XGBoost model
4. gb_model.pkl - Gradient Boosting model
5. catboost_model.pkl - CatBoost model
6. features_in_order.json - Feature names for SHAP
7. feature_engineering.py - Feature engineering module (importable)
8. feature_engineering_func.pkl - Pickled feature engineering function

HOW TO USE IN DEPLOYMENT (Streamlit/app.py):

    import joblib

    # Load model bundle
    bundle = joblib.load('models/stacking_model.pkl')
    model = bundle['model']
    preprocessor = bundle['preprocessor']

    # Load feature engineering function
    engineer_features = joblib.load('models/feature_engineering_func.pkl')

    # Predict on new data (X_raw is a DataFrame with original columns)
    X_eng = engineer_features(X_raw)
    X_transformed = preprocessor.transform(X_eng)
    prediction = model.predict(X_transformed)
    prob_churn = model.predict_proba(X_transformed)[:, 1]
""")

print("\n📊 Model Performance Summary:")
print("-" * 60)
print(f"{'Model':<20} {'Recall':>10} {'Precision':>12} {'F1':>10} {'AUC':>10}")
print("-" * 60)
print(f"{'Stacking (XGB)':<20} {stack_metrics['recall']:>10.4f} {stack_metrics['precision']:>12.4f}{stack_metrics['f1']:>10.4f} {stack_metrics['roc_auc']:>10.4f}")

print(f"{'Random Forest':<20} {rf_metrics['recall']:>10.4f} {rf_metrics['precision']:>12.4f}{rf_metrics['f1']:>10.4f} {rf_metrics['roc_auc']:>10.4f}")

print(f"{'XGBoost':<20} {xgb_metrics['recall']:>10.4f} {xgb_metrics['precision']:>12.4f}{xgb_metrics['f1']:>10.4f} {xgb_metrics['roc_auc']:>10.4f}")

print(f"{'Gradient Boosting':<20} {gb_metrics['recall']:>10.4f} {gb_metrics['precision']:>12.4f}{gb_metrics['f1']:>10.4f} {gb_metrics['roc_auc']:>10.4f}")

print(f"{'CatBoost':<20} {cat_metrics['recall']:>10.4f} {cat_metrics['precision']:>12.4f}{cat_metrics['f1']:>10.4f} {cat_metrics['roc_auc']:>10.4f}")

print("-" * 60)
print("\n✅ All models saved and ready for deployment!")


In [ ]:
# =============================================================================  
# SHAP ANALYSIS FOR STACKING MODEL (XGB META-LEARNER)                                                                           
# FIXED: Ensures feature names match transformed data dimensions
# =============================================================================                                                 
                                                                                                                                
import shap                                                                                                                     
import matplotlib.pyplot as plt                                                                                                 
import seaborn as sns                                                                                                           
import numpy as np                                                                                                              
import pandas as pd                                                                                                             
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score                                     
import warnings                                                                                                                 
warnings.filterwarnings('ignore')                                                                                               
                                                                                                                                
print("=" * 60)                                                                                                                 
print("SHAP ANALYSIS - STACKING MODEL (XGB META)")                                                                              
print("=" * 60)                                                                                                                 
                                                                                                                                
# ============== 1. LOAD SAVED MODEL ==============                                                                             
print("\n[1/7] Loading saved model bundle...")                                                                                  
bundle = joblib.load('models/stacking_model.pkl')                                                                               
model = bundle['model']                                                                                                         
preprocessor = bundle['preprocessor']                                                                                           
feature_names = bundle['feature_names']                                                                                         
                                                                                                                                
# Load feature engineering function                                                                                             
engineer_features = joblib.load('models/feature_engineering_func.pkl')                                                          
print("    ✅ Model and preprocessor loaded")                                                                                   
print(f"    ✅ Features in bundle: {len(feature_names)}")                                                                       
print(f"    Feature names: {feature_names[:5]}... (showing first 5)")                                                           
                                                                                                                                
# ============== 2. PREPARE TEST DATA ==============                                                                            
print("\n[2/7] Preparing test data for SHAP...")                                                                                
                                                                                                                                
# Transform test data                                                                                                           
X_test_eng = engineer_features(X_test)                                                                                          
X_test_transformed = preprocessor.transform(X_test_eng)                                                                         
y_test_array = y_test.values                                                                                                    
                                                                                
print(f"    ✅ Transformed test shape: {X_test_transformed.shape}")                                                             
print(f"    ✅ Churn rate: {y_test_array.mean():.2%}")                                                                          
                                                                                                                                
# Verify feature names match                                                                                                    
if X_test_transformed.shape[1] != len(feature_names):                                                                           
    print(f"\n    ⚠️   WARNING: Mismatch! Data has {X_test_transformed.shape[1]} features, but {len(feature_names)} feature names")       
                                                                                                                       
    print("    Regenerating feature names from preprocessor...")                                                                
                                                                                                                                
    # Get numeric features (after engineering)                                                                                  
    num_features = ['Customer_Age', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon',                     
                    'Contacts_Count_12_mon', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',                  
                    'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'engagement_score']                       
                                                                                                                                
    # Get categorical feature names from preprocessor                                                                           
    cat_encoder = preprocessor.named_transformers_['cat']                                                                       
    cat_features = cat_encoder.get_feature_names_out(['Gender', 'Education_Level', 'Marital_Status', 'Income_Category',         
'Card_Category']).tolist()                                                                                                      
                                                                                                                                
    feature_names = num_features + cat_features                                                                                 
    print(f"    ✅ Fixed: Now using {len(feature_names)} features")                                                             
                                                                                                                                
# ============== 3. MODEL EVALUATION ==============                                                                             
print("\n[3/7] Evaluating model on test set...")                                                                                
y_pred = model.predict(X_test_transformed)                                       
y_prob = model.predict_proba(X_test_transformed)[:, 1]                                                                          
                                                                                                                                
from sklearn.metrics import classification_report, confusion_matrix                                                             
print("\n" + "=" * 60)                                                                                                          
print("MODEL PERFORMANCE ON TEST SET")                                                                                          
print("=" * 60)                                                                                                                 
print(classification_report(y_test, y_pred, target_names=['Retained', 'Attrited']))                                             
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")                                                               
                                                                                                                                
fpr, tpr, _ = roc_curve(y_test, y_prob)                                                                                         
roc_auc_val = auc(fpr, tpr)                                                                                                     
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)                                                       
avg_precision = average_precision_score(y_test, y_prob)                                                                         
                                                                                                                                
print(f"\nROC-AUC: {roc_auc_val:.4f}")                                                                                          
print(f"Average Precision: {avg_precision:.4f}")                                                                                
                                                                                                                                
# ============== 4. CREATE SHAP EXPLAINER ==============                                                                        
print("\n[4/7] Creating SHAP explainer...")                                                                                     
                                                                                                                                
np.random.seed(42)                                                                                                              
sample_size = 100                                                                                                               
background_size = 20                                                                                                            
                                                                                                                                
# Sample test instances                                                                                                         
test_idx = np.random.choice(len(X_test_transformed), sample_size, replace=False)                                                
X_sample = X_test_transformed[test_idx]                                                                                         
y_sample = y_test_array[test_idx]                                                                                               
                                                                                                                                
# Background data                                                                                                               
X_train_eng = engineer_features(X_train)                                                                                        
X_train_transformed = preprocessor.transform(X_train_eng)                                                                       
bg_idx = np.random.choice(len(X_train_transformed), background_size, replace=False)                                             
X_background = X_train_transformed[bg_idx]                                                                                      
                                                                                                                                
print(f"    ✅ Sample: {sample_size} instances, Background: {background_size} instances")                                       
print(f"    ✅ X_sample shape: {X_sample.shape}, X_background shape: {X_background.shape}")                                     
                                                                                                                                
# Create wrapper function                                                                                                       
def model_predict(x):                                                                                                           
    return model.predict_proba(np.atleast_2d(x))[:, 1]                                                                          
                                                                                                                                
# Use KernelExplainer with proper array handling                                                                                
print("    Computing SHAP values...")                                                                                           
explainer = shap.KernelExplainer(model_predict, X_background)                                                                   
shap_values = explainer.shap_values(X_sample, nsamples=100)                                                                     
                                                                                                                                
# Handle different SHAP output formats                                                                                          
if isinstance(shap_values, list):                                                                                               
    shap_values = shap_values[0]  # Take first class for binary classification                                                  
shap_values = np.array(shap_values)                                                                                             
                                                                                                                                
print(f"    ✅ SHAP values shape: {shap_values.shape}")                                                                         
                                                                                                                                
# ============== 5. CALCULATE FEATURE IMPORTANCE ==============                                                                 
print("\n[5/7] Calculating feature importance...")                                                                              
                                                                                                                                
# Ensure SHAP values match feature names                                                                                        
if shap_values.shape[1] != len(feature_names):                                                                                  
    print(f"    ⚠️   SHAP has {shap_values.shape[1]} columns, feature_names has {len(feature_names)}")
    print("    Truncating feature names to match...")                                                                           
    feature_names = feature_names[:shap_values.shape[1]]                                                                        
                                                                                                                                
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)                                                                            
                                                                                                                                
feature_importance = pd.DataFrame({                                                                                             
    'feature': feature_names,                                                                                                   
    'mean_abs_shap': mean_abs_shap                                                                                              
}).sort_values('mean_abs_shap', ascending=False)                                                                                
                                                                                                                                
print("\n" + "=" * 60)                                                           
print("TOP 15 FEATURES BY SHAP IMPORTANCE")                                                                                     
print("=" * 60)                                                                                                                 
print(feature_importance.head(15).to_string(index=False))                                                                       
                                                                                                                                
# ============== 6. SAVE VISUALIZATIONS ==============                                                                          
print("\n[6/7] Saving visualizations...")                                                                                       
                                                                                                                                
# --- Plot 1: SHAP Summary Plot ---                                                                                             
plt.figure(figsize=(14, 10))                                                                                                    
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False, max_display=15)                               
plt.title('SHAP Feature Importance Summary', fontsize=14, fontweight='bold', pad=20)                                            
plt.tight_layout()                                                                                                              
plt.savefig('shap_summary_plot.png', dpi=150, bbox_inches='tight')                                                              
plt.close()                                                                                                                     
print("    ✅ shap_summary_plot.png saved")                                                                                     
                                                                                                                                
# --- Plot 2: SHAP Bar Plot ---                                                                                                 
plt.figure(figsize=(12, 8))                                                                                                     
shap.summary_plot(shap_values, X_sample, feature_names=feature_names,                                                           
                plot_type="bar", show=False, max_display=15)                                                                  
plt.title('Top Features by Mean |SHAP Value|', fontsize=14, fontweight='bold', pad=20)                                          
plt.tight_layout()                                                                                                              
plt.savefig('shap_bar_plot.png', dpi=150, bbox_inches='tight')                                                                  
plt.close()                                                                                                                     
print("    ✅ shap_bar_plot.png saved")                                                                                         
                                                                                                                                
# --- Plot 3: Model Evaluation Plots ---                                                                                        
fig, axes = plt.subplots(1, 3, figsize=(18, 5))                                                                                 
                                                                                                                                
axes[0].hist(y_prob[y_test_array == 0], bins=50, alpha=0.7, label='Retained',                                                   
            color='#2ddbb0', edgecolor='black', linewidth=0.5)                                                                 
axes[0].hist(y_prob[y_test_array == 1], bins=50, alpha=0.7, label='Attrited',                                                   
            color='#ff4d6d', edgecolor='black', linewidth=0.5)                                                                 
axes[0].axvline(x=0.5, color='gold', linestyle='--', linewidth=2, label='Threshold (0.5)')                                      
axes[0].set_xlabel('Predicted Churn Probability', fontsize=11)                                                                  
axes[0].set_ylabel('Count', fontsize=11)                                                                                        
axes[0].set_title('Distribution of Predicted Probabilities', fontsize=12, fontweight='bold')                                    
axes[0].legend(loc='upper right')                                                                                               
axes[0].grid(True, alpha=0.3, linestyle='--')                                                                                   
                                                                                                                                
axes[1].plot(fpr, tpr, color='#e2b96a', lw=2, label=f'ROC Curve (AUC = {roc_auc_val:.4f})')                                     
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)                                                                            
axes[1].set_xlim([0.0, 1.0])                                                                                                    
axes[1].set_ylim([0.0, 1.05])                                                                                                   
axes[1].set_xlabel('False Positive Rate', fontsize=11)                                                                          
axes[1].set_ylabel('True Positive Rate', fontsize=11)                                                                           
axes[1].set_title('ROC Curve', fontsize=12, fontweight='bold')                                                                  
axes[1].legend(loc='lower right')                                                                                               
axes[1].grid(True, alpha=0.3, linestyle='--')                                                                                   
                                                                                                                                
axes[2].plot(recall_curve, precision_curve, color='#4fc8e8', lw=2,                                                              
            label=f'PR Curve (AP = {avg_precision:.4f})')                                                                      
axes[2].set_xlabel('Recall', fontsize=11)                                                                                       
axes[2].set_ylabel('Precision', fontsize=11)   

In [ ]:
# =============================================================================
# SHAP ANALYSIS FOR STACKING MODEL - FIXED VERSION                                                                              
# Prints all features and ensures proper alignment            
# =============================================================================                                                 
                                                                                                                                
import shap                                                                                                                     
import matplotlib.pyplot as plt                                                                                                 
import numpy as np                                                                                                              
import pandas as pd                                                                                                             
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score                                     
import warnings                                                                                                                 
warnings.filterwarnings('ignore')                                                                                               
                                                                                                                                
print("=" * 60)                                                                                                                 
print("SHAP ANALYSIS - STACKING MODEL")                                                                                         
print("=" * 60)                                                                                                                 
                                                                                                                                
# ============== 1. LOAD MODEL AND INSPECT ==============                                                                       
print("\n[1/8] Loading model bundle and inspecting features...")                                                                
bundle = joblib.load('models/stacking_model.pkl')                                                                               
model = bundle['model']                                                                                                         
preprocessor = bundle['preprocessor']                                                                                           
saved_feature_names = bundle['feature_names']                                                                                   
                                                                                                                                
# Load feature engineering function                                                                                             
engineer_features = joblib.load('models/feature_engineering_func.pkl')                                                          
                                                                                                                                
print(f"    Features in saved bundle: {len(saved_feature_names)}")                                                              
print("\n📋 ALL FEATURES IN MODEL:")                                                                                            
print("-" * 70)                                                                                                                 
for i, feat in enumerate(saved_feature_names):                                                                                  
    print(f"  {i+1:2d}. {feat}")                                                                                                
print("-" * 70)                                                                                                                 
                                                                                                                                
# ============== 2. GET ACTUAL FEATURE NAMES FROM PREPROCESSOR ==============                                                   
print("\n[2/8] Getting actual feature names from preprocessor output...")                                                       
                                                                                                                                
# Numeric features (original + engineered)                                                                                      
numeric_original = ['Customer_Age', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon',                     
                    'Contacts_Count_12_mon', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',                  
                    'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']                                           
engineered_features = ['engagement_score']  # Only this one, avg_amt_per_txn is not used in final model                         
num_feature_names = numeric_original + engineered_features                                                                      
                                                                                                                                
# Get categorical feature names from the fitted preprocessor                                                                    
categorical_features = ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']                      
cat_encoder = preprocessor.named_transformers_['cat']                                                                           
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features).tolist()                                            
                                                                                                                                
# Full feature names                                                                                                            
actual_feature_names = num_feature_names + cat_feature_names                                                                    
                                                                                                                                
print(f"    Numeric features: {len(num_feature_names)}")                                                                        
print(f"    Categorical features (after OneHot): {len(cat_feature_names)}")                                                     
print(f"    Total: {len(actual_feature_names)}")                                                                                
                                                                                                                                
# Use actual feature names from preprocessor                                                                                    
feature_names = actual_feature_names                                                                                            
print(f"\n    ✅ Using {len(feature_names)} features for SHAP analysis")                                                        
                                                                                                                                
# ============== 3. PREPARE TEST DATA ==============                                                                            
print("\n[3/8] Preparing test data...")                                                                                         
                                                                                                                                
X_test_eng = engineer_features(X_test)                                                                                          
X_test_transformed = preprocessor.transform(X_test_eng)                                                                         
y_test_array = y_test.values                                                                                                    
                                                                                                                                
print(f"    ✅ Transformed test data shape: {X_test_transformed.shape}")                                                        
print(f"    ✅ Matches feature names: {X_test_transformed.shape[1] == len(feature_names)}")                                     
                                                                                                                                
# ============== 4. MODEL EVALUATION ==============                                                                             
print("\n[4/8] Evaluating model...")                                                                                            
y_pred = model.predict(X_test_transformed)                                                                                      
y_prob = model.predict_proba(X_test_transformed)[:, 1]                                                                          
                                                                                                                                
from sklearn.metrics import classification_report, confusion_matrix                                                             
print("\n" + "=" * 60)                                                                                                          
print("MODEL PERFORMANCE")                                                                                                      
print("=" * 60)                                                                                                                 
print(classification_report(y_test, y_pred, target_names=['Retained', 'Attrited']))                                             
                                                                                                                                
fpr, tpr, _ = roc_curve(y_test, y_prob)                                                                                         
roc_auc_val = auc(fpr, tpr)                                                                                                     
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)                                                       
avg_precision = average_precision_score(y_test, y_prob)                                                                         
print(f"\nROC-AUC: {roc_auc_val:.4f} | Average Precision: {avg_precision:.4f}")                                                 
                                                                                                                                
# ============== 5. CREATE SHAP EXPLAINER ==============                                                                        
print("\n[5/8] Creating SHAP explainer...")                                                                                     
                                                                                                                                
np.random.seed(42)                                                                                                              
sample_size = 50                                                                                                                
background_size = 20                                                                                                            
                                                                                                                                
# Sample indices                                                                                                                
test_idx = np.random.choice(len(X_test_transformed), sample_size, replace=False)                                                
X_sample = X_test_transformed[test_idx]                                                                                         
                                                                                                                                
# Background from training                                                                                                      
X_train_eng = engineer_features(X_train)                                                                                        
X_train_transformed = preprocessor.transform(X_train_eng)                                                                       
bg_idx = np.random.choice(len(X_train_transformed), background_size, replace=False)                                             
X_background = X_train_transformed[bg_idx]                                                                                      
                                                                                                                                
print(f"    Sample shape: {X_sample.shape}")                                                                                    
print(f"    Background shape: {X_background.shape}")                                                                            
                                                                                                                                
# Prediction wrapper                                                                                                            
def predict_fn(x):                                                                                                              
    return model.predict_proba(np.atleast_2d(x))[:, 1]                                                                          
                                                                                                                                
# Use KernelExplainer with proper shapes                                                                                        
print("    Computing SHAP values...")                                                                                           
explainer = shap.KernelExplainer(predict_fn, X_background)                                                                      
shap_values = explainer.shap_values(X_sample, nsamples=100)                                                                     
                                                                                                                                
# Handle different SHAP output formats                                                                                          
if isinstance(shap_values, list):                                                                                               
    shap_values = np.array(shap_values[0])  # Binary classification                                                             
elif isinstance(shap_values, dict):                                                                                             
    shap_values = np.array(shap_values[1])  # Class 1 values                                                                    
else:                                                                                                                           
    shap_values = np.array(shap_values)                                                                                         
                                                                                                                                
print(f"    ✅ SHAP values shape: {shap_values.shape}")                                                                         
print(f"    ✅ Expected: ({sample_size}, {len(feature_names)})")                                                                
                                                                                                                                
# ============== 6. FEATURE IMPORTANCE ==============                                                                           
print("\n[6/8] Calculating feature importance...")                                                                              
                                                                                                                                
# Ensure shapes match                                                                                                           
if shap_values.shape[1] != len(feature_names):                                                                                  
    print(f"    ⚠️   Shape mismatch! SHAP: {shap_values.shape[1]}, Features: {len(feature_names)}")                              
    print("    Truncating features to match...")                                                                                
    feature_names = feature_names[:shap_values.shape[1]]                                                                        
                                                                                                                                
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)                                                                            
                                                                                                                                
feature_importance = pd.DataFrame({                                                                                             
    'feature': feature_names,                                                                                                   
    'mean_abs_shap': mean_abs_shap                                                                                              
}).sort_values('mean_abs_shap', ascending=False)                                                                                
                                                                                                                                
print("\n" + "=" * 60)                                                           
print("ALL FEATURES BY SHAP IMPORTANCE")                                                                                        
print("=" * 60)                                                                                                                 
for i, row in feature_importance.iterrows():                                                                                    
    print(f"  {i+1:2d}. {row['feature']:<45} {row['mean_abs_shap']:.5f}")                                                       
                                                                                                                                
# ============== 7. SAVE PLOTS ==============                                                                                   
print("\n[7/8] Saving visualizations...")                                                                                       
                                                                                                                                
# SHAP Summary Plot                                                                                                             
plt.figure(figsize=(14, 12))                                                                                                    
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False, max_display=len(feature_names))               
plt.title('SHAP Feature Importance Summary', fontsize=14, fontweight='bold', pad=20)                                            
plt.tight_layout()                                                                                                              
plt.savefig('shap_summary_plot.png', dpi=150, bbox_inches='tight')    
plt.show()                                                          
plt.close()                                                                                                                     
print("    ✅ shap_summary_plot.png saved")                                                                                     
                                                                                                                                
# SHAP Bar Plot                                                                                                                 
plt.figure(figsize=(12, 10))                                                                                                    
shap.summary_plot(shap_values, X_sample, feature_names=feature_names,                                                           
                plot_type="bar", show=False, max_display=len(feature_names))                                                  
plt.title('Top Features by Mean |SHAP Value|', fontsize=14, fontweight='bold', pad=20)                                          
plt.tight_layout()                                                                                                              
plt.savefig('shap_bar_plot.png', dpi=150, bbox_inches='tight')
plt.show()                                                                
plt.close()                                                                                                                     
print("    ✅ shap_bar_plot.png saved")                                                                                         
                                                                                                                                
# Model Evaluation Plots                                                                                                        
fig, axes = plt.subplots(1, 3, figsize=(18, 5))                                                                                 
                                                                                                                                
axes[0].hist(y_prob[y_test_array == 0], bins=50, alpha=0.7, label='Retained', color='#2ddbb0')                                  
axes[0].hist(y_prob[y_test_array == 1], bins=50, alpha=0.7, label='Attrited', color='#ff4d6d')                                  
axes[0].axvline(x=0.5, color='gold', linestyle='--', linewidth=2)                                                               
axes[0].set_xlabel('Predicted Probability')                                                                                     
axes[0].set_title('Probability Distribution')                                                                                   
axes[0].legend()                                                                                                                
axes[0].grid(True, alpha=0.3)                                                                                                   
                                                                                                                                
axes[1].plot(fpr, tpr, color='#e2b96a', lw=2, label=f'AUC = {roc_auc_val:.4f}')                                                 
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)                                                                                       
axes[1].set_xlabel('False Positive Rate')                                                                                       
axes[1].set_ylabel('True Positive Rate')                                                                                        
axes[1].set_title('ROC Curve')                                                                                                  
axes[1].legend()                                                                                                                
axes[1].grid(True, alpha=0.3)                                                                                                   
                                                                                                                                
axes[2].plot(recall_curve, precision_curve, color='#4fc8e8', lw=2, label=f'AP = {avg_precision:.4f}')                           
axes[2].set_xlabel('Recall')                                                                                                    
axes[2].set_ylabel('Precision')                                                                                                 
axes[2].set_title('Precision-Recall Curve')                                                                                     
axes[2].legend()                                                                                                                
axes[2].grid(True, alpha=0.3)                                                                                                   
                                                                                                                                
plt.tight_layout()                                                                                                              
plt.savefig('model_evaluation_plots.png', dpi=150, bbox_inches='tight')                                                         
plt.close()                                                                                                                     
print("    ✅ model_evaluation_plots.png saved")                                                                                
                                                                                                                                
# Save SHAP data                                                                                                                
shap_data = {                                                                                                                   
    'feature_importance': feature_importance,                                                                                   
    'shap_values': shap_values,                                                                                                 
    'feature_names': feature_names,                                                                                             
    'mean_abs_shap': mean_abs_shap,                                                                                             
    'X_sample': X_sample,                                                                                                       
    'sample_indices': test_idx                                                                                                  
}                                                                                                                               
joblib.dump(shap_data, 'models/shap_data.pkl')                                                                                  
print("    ✅ models/shap_data.pkl saved")                                                                                      
                                                                                                                                
feature_importance.to_csv('shap_feature_importance.csv', index=False)                                                           
print("    ✅ shap_feature_importance.csv saved")                                                                               
                                                                                                                                
# ============== SUMMARY ==============                                                                                         
print("\n" + "=" * 60)                                                                                                          
print("SHAP ANALYSIS COMPLETE")                                                                                                 
print("=" * 60)                                                                                                                 
print(f"\nTotal features analyzed: {len(feature_names)}")                                                                       
print(f"SHAP values shape: {shap_values.shape}")                                                                                
print("\n✅ All artifacts saved successfully!")    

In [ ]:
# =============================================================================
# SHAP ANALYSIS FOR STACKING MODEL - FIXED VERSION                                                                              
# Prints all features and ensures proper alignment            
# =============================================================================                                                 
                                                                                                                                
import shap                                                                                                                     
import matplotlib.pyplot as plt                                                                                                 
import numpy as np                                                                                                              
import pandas as pd                                                                                                             
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score                                     
import warnings                                                                                                                 
warnings.filterwarnings('ignore')                                                                                               
                                                                                                                                
print("=" * 60)                                                                                                                 
print("SHAP ANALYSIS - STACKING MODEL")                                                                                         
print("=" * 60)                                                                                                                 
                                                                                                                                
# ============== 1. LOAD MODEL AND INSPECT ==============                                                                       
print("\n[1/8] Loading model bundle and inspecting features...")                                                                
bundle = joblib.load('models/stacking_model.pkl')                                                                               
model = bundle['model']                                                                                                         
preprocessor = bundle['preprocessor']                                                                                           
saved_feature_names = bundle['feature_names']                                                                                   
                                                                                                                                
# Load feature engineering function                                                                                             
engineer_features = joblib.load('models/feature_engineering_func.pkl')                                                          
                                                                                                                                
print(f"    Features in saved bundle: {len(saved_feature_names)}")                                                              
print("\n📋 ALL FEATURES IN MODEL:")                                                                                            
print("-" * 70)                                                                                                                 
for i, feat in enumerate(saved_feature_names):                                                                                  
    print(f"  {i+1:2d}. {feat}")                                                                                                
print("-" * 70)                                                                                                                 
                                                                                                                                
# ============== 2. GET ACTUAL FEATURE NAMES FROM PREPROCESSOR ==============                                                   
print("\n[2/8] Getting actual feature names from preprocessor output...")                                                       
                                                                                                                                
# Numeric features (original + engineered)                                                                                      
numeric_original = ['Customer_Age', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon',                     
                    'Contacts_Count_12_mon', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',                  
                    'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']                                           
engineered_features = ['engagement_score']  # Only this one, avg_amt_per_txn is not used in final model                         
num_feature_names = numeric_original + engineered_features                                                                      
                                                                                                                                
# Get categorical feature names from the fitted preprocessor                                                                    
categorical_features = ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']                      
cat_encoder = preprocessor.named_transformers_['cat']                                                                           
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features).tolist()                                            
                                                                                                                                
# Full feature names                                                                                                            
actual_feature_names = num_feature_names + cat_feature_names                                                                    
                                                                                                                                
print(f"    Numeric features: {len(num_feature_names)}")                                                                        
print(f"    Categorical features (after OneHot): {len(cat_feature_names)}")                                                     
print(f"    Total: {len(actual_feature_names)}")                                                                                
                                                                                                                                
# Use actual feature names from preprocessor                                                                                    
feature_names = actual_feature_names                                                                                            
print(f"\n    ✅ Using {len(feature_names)} features for SHAP analysis")                                                        
                                                                                                                                
# ============== 3. PREPARE TEST DATA ==============                                                                            
print("\n[3/8] Preparing test data...")                                                                                         
                                                                                                                                
X_test_eng = engineer_features(X_test)                                                                                          
X_test_transformed = preprocessor.transform(X_test_eng)                                                                         
y_test_array = y_test.values                                                                                                    
                                                                                                                                
print(f"    ✅ Transformed test data shape: {X_test_transformed.shape}")                                                        
print(f"    ✅ Matches feature names: {X_test_transformed.shape[1] == len(feature_names)}")                                     
                                                                                                                                
# ============== 4. MODEL EVALUATION ==============                                                                             
print("\n[4/8] Evaluating model...")                                                                                            
y_pred = model.predict(X_test_transformed)                                                                                      
y_prob = model.predict_proba(X_test_transformed)[:, 1]                                                                          
                                                                                                                                
from sklearn.metrics import classification_report, confusion_matrix                                                             
print("\n" + "=" * 60)                                                                                                          
print("MODEL PERFORMANCE")                                                                                                      
print("=" * 60)                                                                                                                 
print(classification_report(y_test, y_pred, target_names=['Retained', 'Attrited']))                                             
                                                                                                                                
fpr, tpr, _ = roc_curve(y_test, y_prob)                                                                                         
roc_auc_val = auc(fpr, tpr)                                                                                                     
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)                                                       
avg_precision = average_precision_score(y_test, y_prob)                                                                         
print(f"\nROC-AUC: {roc_auc_val:.4f} | Average Precision: {avg_precision:.4f}")                                                 
                                                                                                                                
# ============== 5. CREATE SHAP EXPLAINER ==============                                                                        
print("\n[5/8] Creating SHAP explainer...")                                                                                     
                                                                                                                                
np.random.seed(42)                                                                                                              
sample_size = 50                                                                                                                
background_size = 20                                                                                                            
                                                                                                                                
# Sample indices                                                                                                                
test_idx = np.random.choice(len(X_test_transformed), sample_size, replace=False)                                                
X_sample = X_test_transformed[test_idx]                                                                                         
                                                                                                                                
# Background from training                                                                                                      
X_train_eng = engineer_features(X_train)                                                                                        
X_train_transformed = preprocessor.transform(X_train_eng)                                                                       
bg_idx = np.random.choice(len(X_train_transformed), background_size, replace=False)                                             
X_background = X_train_transformed[bg_idx]                                                                                      
                                                                                                                                
print(f"    Sample shape: {X_sample.shape}")                                                                                    
print(f"    Background shape: {X_background.shape}")                                                                            
                                                                                                                                
# Prediction wrapper                                                                                                            
def predict_fn(x):                                                                                                              
    return model.predict_proba(np.atleast_2d(x))[:, 1]                                                                          
                                                                                                                                
# Use KernelExplainer with proper shapes                                                                                        
print("    Computing SHAP values...")                                                                                           
explainer = shap.KernelExplainer(predict_fn, X_background)                                                                      
shap_values = explainer.shap_values(X_sample, nsamples=100)                                                                     
                                                                                                                                
# Handle different SHAP output formats                                                                                          
if isinstance(shap_values, list):                                                                                               
    shap_values = np.array(shap_values[0])  # Binary classification                                                             
elif isinstance(shap_values, dict):                                                                                             
    shap_values = np.array(shap_values[1])  # Class 1 values                                                                    
else:                                                                                                                           
    shap_values = np.array(shap_values)                                                                                         
                                                                                                                                
print(f"    ✅ SHAP values shape: {shap_values.shape}")                                                                         
print(f"    ✅ Expected: ({sample_size}, {len(feature_names)})")                                                                
                                                                                                                                
# ============== 6. FEATURE IMPORTANCE ==============                                                                           
print("\n[6/8] Calculating feature importance...")                                                                              
                                                                                                                                
# Ensure shapes match                                                                                                           
if shap_values.shape[1] != len(feature_names):                                                                                  
    print(f"    ⚠️   Shape mismatch! SHAP: {shap_values.shape[1]}, Features: {len(feature_names)}")                              
    print("    Truncating features to match...")                                                                                
    feature_names = feature_names[:shap_values.shape[1]]                                                                        
                                                                                                                                
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)                                                                            
                                                                                                                                
feature_importance = pd.DataFrame({                                                                                             
    'feature': feature_names,                                                                                                   
    'mean_abs_shap': mean_abs_shap                                                                                              
}).sort_values('mean_abs_shap', ascending=False)                                                                                
                                                                                                                                
print("\n" + "=" * 60)                                                           
print("ALL FEATURES BY SHAP IMPORTANCE")                                                                                        
print("=" * 60)                                                                                                                 
for i, row in feature_importance.iterrows():                                                                                    
    print(f"  {i+1:2d}. {row['feature']:<45} {row['mean_abs_shap']:.5f}")                                                       
                                                                                                                                
# ============== 7. SAVE PLOTS ==============                                                                                   
print("\n[7/8] Saving visualizations...")                                                                                       
                                                                                                                                
# SHAP Summary Plot                                                                                                             
plt.figure(figsize=(14, 12))                                                                                                    
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False, max_display=len(feature_names))               
plt.title('SHAP Feature Importance Summary', fontsize=14, fontweight='bold', pad=20)                                            
plt.tight_layout()                                                                                                              
plt.savefig('shap_summary_plot.png', dpi=150, bbox_inches='tight')    
plt.show()                                                          
plt.close()                                                                                                                     
print("    ✅ shap_summary_plot.png saved")                                                                                     
                                                                                                                                
# SHAP Bar Plot                                                                                                                 
plt.figure(figsize=(12, 10))                                                                                                    
shap.summary_plot(shap_values, X_sample, feature_names=feature_names,                                                           
                plot_type="bar", show=False, max_display=len(feature_names))                                                  
plt.title('Top Features by Mean |SHAP Value|', fontsize=14, fontweight='bold', pad=20)                                          
plt.tight_layout()                                                                                                              
plt.savefig('shap_bar_plot.png', dpi=150, bbox_inches='tight')
plt.show()                                                                
plt.close()                                                                                                                     
print("    ✅ shap_bar_plot.png saved")                                                                                         
                                                                                                                                
# Model Evaluation Plots                                                                                                        
fig, axes = plt.subplots(1, 3, figsize=(18, 5))                                                                                 
                                                                                                                                
axes[0].hist(y_prob[y_test_array == 0], bins=50, alpha=0.7, label='Retained', color='#2ddbb0')                                  
axes[0].hist(y_prob[y_test_array == 1], bins=50, alpha=0.7, label='Attrited', color='#ff4d6d')                                  
axes[0].axvline(x=0.5, color='gold', linestyle='--', linewidth=2)                                                               
axes[0].set_xlabel('Predicted Probability')                                                                                     
axes[0].set_title('Probability Distribution')                                                                                   
axes[0].legend()                                                                                                                
axes[0].grid(True, alpha=0.3)                                                                                                   
                                                                                                                                
axes[1].plot(fpr, tpr, color='#e2b96a', lw=2, label=f'AUC = {roc_auc_val:.4f}')                                                 
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)                                                                                       
axes[1].set_xlabel('False Positive Rate')                                                                                       
axes[1].set_ylabel('True Positive Rate')                                                                                        
axes[1].set_title('ROC Curve')                                                                                                  
axes[1].legend()                                                                                                                
axes[1].grid(True, alpha=0.3)                                                                                                   
                                                                                                                                
axes[2].plot(recall_curve, precision_curve, color='#4fc8e8', lw=2, label=f'AP = {avg_precision:.4f}')                           
axes[2].set_xlabel('Recall')                                                                                                    
axes[2].set_ylabel('Precision')                                                                                                 
axes[2].set_title('Precision-Recall Curve')                                                                                     
axes[2].legend()                                                                                                                
axes[2].grid(True, alpha=0.3)                                                                                                   
                                                                                                                                
plt.tight_layout()                                                                                                              
plt.savefig('model_evaluation_plots.png', dpi=150, bbox_inches='tight')                                                         
plt.close()                                                                                                                     
print("    ✅ model_evaluation_plots.png saved")                                                                                
                                                                                                                                
# Save SHAP data                                                                                                                
shap_data = {                                                                                                                   
    'feature_importance': feature_importance,                                                                                   
    'shap_values': shap_values,                                                                                                 
    'feature_names': feature_names,                                                                                             
    'mean_abs_shap': mean_abs_shap,                                                                                             
    'X_sample': X_sample,                                                                                                       
    'sample_indices': test_idx                                                                                                  
}                                                                                                                               
joblib.dump(shap_data, 'models/shap_data.pkl')                                                                                  
print("    ✅ models/shap_data.pkl saved")                                                                                      
                                                                                                                                
feature_importance.to_csv('shap_feature_importance.csv', index=False)                                                           
print("    ✅ shap_feature_importance.csv saved")                                                                               
                                                                                                                                
# ============== SUMMARY ==============                                                                                         
print("\n" + "=" * 60)                                                                                                          
print("SHAP ANALYSIS COMPLETE")                                                                                                 
print("=" * 60)                                                                                                                 
print(f"\nTotal features analyzed: {len(feature_names)}")                                                                       
print(f"SHAP values shape: {shap_values.shape}")                                                                                
print("\n✅ All artifacts saved successfully!") 